# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hrushi56/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
!pip -q install duckdb pandas pyarrow huggingface_hub

In [2]:
import duckdb
import pandas as pd
import os

con = duckdb.connect()

con.sql("""
INSTALL httpfs;
LOAD httpfs;
""")

In [3]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con.execute(f"""
CREATE OR REPLACE SECRET hf_secret(
TYPE HUGGINGFACE,
TOKEN '{HF_TOKEN}'
);
""")

print("Connected!")

Connected!


## 1. My rule and its reason codes

My Rule

This baseline rule identifies pages that may benefit from a content refresh. Pages with many impressions but relatively few clicks and average search positions between 5 and 20 are considered good refresh opportunities. These pages are already visible in search results but may not be attracting enough user clicks.

| Reason Code       | Meaning                                      |
| ----------------- | -------------------------------------------- |
| LOW_CTR           | High impressions but low clicks.             |
| MID_POSITION      | Average search position between 5 and 20.    |
| REFRESH_CANDIDATE | Page should be reviewed for content updates. |


In [4]:
signal_check = con.sql("""
SELECT
CASE
WHEN gsc_avg_position < 5 THEN 'Top 5'
WHEN gsc_avg_position < 10 THEN '5-10'
WHEN gsc_avg_position < 20 THEN '10-20'
ELSE '20+'
END AS position_bucket,

COUNT(*) AS n,
AVG(gsc_clicks) AS avg_clicks,
AVG(gsc_impressions) AS avg_impressions

FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
)

WHERE
gsc_data_available IS TRUE

GROUP BY position_bucket
ORDER BY avg_clicks DESC;
""").df()

signal_check


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,position_bucket,n,avg_clicks,avg_impressions
0,Top 5,1222668,0.364332,99.481697
1,5-10,932273,0.220486,75.216431
2,10-20,543782,0.170351,54.161756
3,20+,912338,0.085701,65.162325


## 2. Build the ranked queue (writes the CSV)

## Baseline Scoring Rule

A refresh score is calculated using historical search performance. Pages with high impressions, low clicks, and average search positions between 5 and 20 receive higher scores because they are visible in search results but may benefit from updated content.

Action Label:
- Refresh Content

Reason Code:
- LOW_CTR

In [5]:
import os

baseline_df = con.sql("""
SELECT
    report_date,
    content_hash_id,

    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,

    CASE
        WHEN gsc_avg_position BETWEEN 5 AND 20
             AND gsc_impressions > 100
             AND gsc_clicks < 20
        THEN
            (gsc_impressions * 0.6)
            - (gsc_clicks * 2)
            + (20 - gsc_avg_position)

        ELSE 0
    END AS baseline_score,

    'LOW_CTR' AS reason_code,

    'Refresh Content' AS action

FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
)

WHERE gsc_data_available IS TRUE

ORDER BY baseline_score DESC
""").df()

baseline_df.head(20)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,baseline_score,reason_code,action
0,2026-03-04,content_945d6ff91386c817,37368,0,8.613948,22432.186052,LOW_CTR,Refresh Content
1,2026-03-15,content_1642f339bd6e7c8d,16454,1,5.131579,9885.268421,LOW_CTR,Refresh Content
2,2026-03-28,content_046fc480045b88f5,14185,1,7.050335,8521.949665,LOW_CTR,Refresh Content
3,2026-03-29,content_046fc480045b88f5,13764,0,6.915650,8271.484350,LOW_CTR,Refresh Content
4,2026-03-02,content_7c6373141eae744a,12648,4,6.159235,7594.640765,LOW_CTR,Refresh Content
5,2026-03-04,content_0bca6d9a85a9b408,11464,0,7.485694,6890.914306,LOW_CTR,Refresh Content
6,2026-03-30,content_046fc480045b88f5,10895,1,7.697292,6547.302708,LOW_CTR,Refresh Content
7,2026-03-09,content_7c6373141eae744a,9987,3,7.245319,5998.954681,LOW_CTR,Refresh Content
8,2026-03-13,content_65c75874a23fca87,9885,1,7.168842,5941.831158,LOW_CTR,Refresh Content
9,2026-03-03,content_e578ac84778da489,9842,11,5.430096,5897.769904,LOW_CTR,Refresh Content


In [6]:
os.makedirs("work/outputs", exist_ok=True)

baseline_df.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("✅ CSV written successfully!")

print(baseline_df.head(10))

✅ CSV written successfully!
  report_date           content_hash_id  gsc_impressions  gsc_clicks  \
0  2026-03-04  content_945d6ff91386c817            37368           0   
1  2026-03-15  content_1642f339bd6e7c8d            16454           1   
2  2026-03-28  content_046fc480045b88f5            14185           1   
3  2026-03-29  content_046fc480045b88f5            13764           0   
4  2026-03-02  content_7c6373141eae744a            12648           4   
5  2026-03-04  content_0bca6d9a85a9b408            11464           0   
6  2026-03-30  content_046fc480045b88f5            10895           1   
7  2026-03-09  content_7c6373141eae744a             9987           3   
8  2026-03-13  content_65c75874a23fca87             9885           1   
9  2026-03-03  content_e578ac84778da489             9842          11   

   gsc_avg_position  baseline_score reason_code           action  
0          8.613948    22432.186052     LOW_CTR  Refresh Content  
1          5.131579     9885.268421     LOW_C

## 3. Top-20 review

## Top-20 Review

The table below reviews the highest-ranked refresh candidates generated by the baseline rule. Each recommendation includes the suggested action, the reason code, a confidence note, and one possible reason why the recommendation could be incorrect.

In [7]:
top20 = baseline_df.head(20).copy()

top20["confidence_note"] = "Medium confidence - based on historical search performance."

top20["what_would_make_it_wrong"] = (
    "Seasonal traffic, recent content updates, or temporary ranking changes."
)

review = top20[
    [
        "content_hash_id",
        "baseline_score",
        "action",
        "reason_code",
        "confidence_note",
        "what_would_make_it_wrong",
    ]
]

display(review)

,content_hash_id,baseline_score,action,reason_code,confidence_note,what_would_make_it_wrong
0,content_945d6ff91386c817,22432.186052,Refresh Content,LOW_CTR,Medium confidence - based on historical search...,"Seasonal traffic, recent content updates, or t..."
1,content_1642f339bd6e7c8d,9885.268421,Refresh Content,LOW_CTR,Medium confidence - based on historical search...,"Seasonal traffic, recent content updates, or t..."
2,content_046fc480045b88f5,8521.949665,Refresh Content,LOW_CTR,Medium confidence - based on historical search...,"Seasonal traffic, recent content updates, or t..."
3,content_046fc480045b88f5,8271.484350,Refresh Content,LOW_CTR,Medium confidence - based on historical search...,"Seasonal traffic, recent content updates, or t..."
4,content_7c6373141eae744a,7594.640765,Refresh Content,LOW_CTR,Medium confidence - based on historical search...,"Seasonal traffic, recent content updates, or t..."
5,content_0bca6d9a85a9b408,6890.914306,Refresh Content,LOW_CTR,Medium confidence - based on historical search...,"Seasonal traffic, recent content updates, or t..."
6,content_046fc480045b88f5,6547.302708,Refresh Content,LOW_CTR,Medium confidence - based on historical search...,"Seasonal traffic, recent content updates, or t..."
7,content_7c6373141eae744a,5998.954681,Refresh Content,LOW_CTR,Medium confidence - based on historical search...,"Seasonal traffic, recent content updates, or t..."
8,content_65c75874a23fca87,5941.831158,Refresh Content,LOW_CTR,Medium confidence - based on historical search...,"Seasonal traffic, recent content updates, or t..."
9,content_e578ac84778da489,5897.769904,Refresh Content,LOW_CTR,Medium confidence - based on historical search...,"Seasonal traffic, recent content updates, or t..."


## 4. Weak picks + leakage check

## Weak Picks and Leakage Check

The lowest-ranked pages are unlikely to require immediate refresh because they do not satisfy the baseline rule.

Only historical Search Console metrics (impressions, clicks, and average position) were used. No future-window information or label-derived features were included, so no data leakage was introduced.

In [8]:
print("Lowest Scoring Pages")

display(
    baseline_df.sort_values("baseline_score").head(10)
)

print("\nLeakage Check")

print("✓ No future-window features used")
print("✓ No label-derived columns used")
print("✓ Only historical metrics included")

Lowest Scoring Pages


,report_date,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,baseline_score,reason_code,action
2407377,2026-03-10,content_b8465fead1d8e67d,376,2,23.234043,0.0,LOW_CTR,Refresh Content
2407364,2026-03-10,content_9e8f6a0c53683bd7,2,0,7.500000,0.0,LOW_CTR,Refresh Content
2407365,2026-03-10,content_82c0af915bb94fa6,36,1,2.527778,0.0,LOW_CTR,Refresh Content
2407366,2026-03-10,content_fc8fa2ce851f8f52,26,0,6.461538,0.0,LOW_CTR,Refresh Content
2407367,2026-03-10,content_75197363ba40652f,55,0,23.600000,0.0,LOW_CTR,Refresh Content
2407368,2026-03-10,content_dac53d93054c08a3,478,3,38.073222,0.0,LOW_CTR,Refresh Content
2407369,2026-03-10,content_24007ac44b9d1661,129,0,21.651163,0.0,LOW_CTR,Refresh Content
2407370,2026-03-10,content_1612bddc993ce606,41,0,0.926829,0.0,LOW_CTR,Refresh Content
2407371,2026-03-10,content_9c3282100e77ab5d,298,0,36.892617,0.0,LOW_CTR,Refresh Content
2407372,2026-03-10,content_a4aacad39507fbb7,140,0,4.421429,0.0,LOW_CTR,Refresh Content



Leakage Check
✓ No future-window features used
✓ No label-derived columns used
✓ Only historical metrics included


## Self-check

Before you submit, confirm each line honestly:

- ✅ Every section above is filled — markdown thinking AND the code that backs it
- ✅ The notebook runs top to bottom with no errors (Runtime → Run all)
- ✅ No client names, URLs, or private queries anywhere
- ✅ My claims use careful words: observed, measured, directional, decision-support
- ✅ Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.